# 📈 02 — VAR-GFEVD Statistical Spillover Baseline

This research notebook establishes the econometric baseline for **CryptoGraph-Secure** using Vector Autoregressions (VAR) and Forecast Error Variance Decomposition (FEVD):

1. **Econometric Framework:** Implementation of the **Diebold-Yilmaz (2012)** Spillover Index using an order-invariant **Generalized Forecast Error Variance Decomposition (GFEVD)** based on Pesaran & Shin (1998).
2. **Total Spillover Index (TSI):** Quantifying what percentage of systemic return variance across the frozen 20-coin crypto market is driven by cross-asset contagion.
3. **Net Directional Connectedness Ranking:** Identifying which assets are net systemic transmitters of volatility vs. net shock receivers.
4. **Econometric Baseline Export:** Exporting [`data/processed/statistical_connectedness.csv`](../data/processed/statistical_connectedness.csv) to benchmark against `TempME` graph attention in Notebook 03.

In [5]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

# Add src directory to path
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import torch
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

pio.templates.default = "plotly_dark"

from cryptograph.data.dataloader import load_processed_dataset
from cryptograph.models.baselines.var_spillover import VARSpilloverBaseline

print("✅ Econometric modeling and baseline libraries loaded successfully!")


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
✅ Econometric modeling and baseline libraries loaded successfully!


## 1. Load Processed Dataset & Extract Common Active Period

We load `data/processed/crypto_graph_dataset.pt` and isolate the period where all 20 cryptocurrencies are simultaneously active (2,131 trading days). A balanced panel is required to compute stationary VAR coefficients without missing-data imputation artifacts.


In [6]:
dataset_path = Path("../data/processed/crypto_graph_dataset.pt")
data = load_processed_dataset(dataset_path)

features = data["features"]
active_mask = data["active_mask"]
asset_names = data["asset_names"]
timestamps = pd.to_datetime(data["timestamps"])

# Extract daily log returns
log_returns_df = pd.DataFrame(
    features[:, :, 0].numpy(),
    index=timestamps,
    columns=asset_names
)

# Common active mask
all_active_mask = active_mask.numpy().all(axis=1)
ret_active = log_returns_df.loc[all_active_mask]

print(f"📊 Total Assets:         {len(asset_names)} cryptocurrencies")
print(f"📈 Common Active Window: {ret_active.index[0].strftime('%Y-%m-%d')} to {ret_active.index[-1].strftime('%Y-%m-%d')}")
print(f"⏱️ Sample Length:        {len(ret_active)} consecutive trading days")


📊 Total Assets:         20 cryptocurrencies
📈 Common Active Window: 2020-10-14 to 2026-09-24
⏱️ Sample Length:        2131 consecutive trading days


## 2. Pairwise Correlation Heatmap & Inter-Asset Dependence

Before modeling directional spillover transmission, we inspect the static pairwise correlation matrix across all 20 cryptocurrencies.


In [7]:
corr_matrix = ret_active.corr()

fig_corr = px.imshow(
    corr_matrix,
    x=asset_names,
    y=asset_names,
    color_continuous_scale="Viridis",
    zmin=0.0,
    zmax=1.0,
    title="Pairwise Return Correlation Matrix (20 Cryptocurrencies)",
    labels=dict(color="Correlation")
)
fig_corr.update_layout(width=720, height=680)
fig_corr.show()

off_diag_vals = corr_matrix.values[~np.eye(len(corr_matrix), dtype=bool)]
print(f"🔗 Mean Inter-Asset Correlation: {off_diag_vals.mean():.3f} (Min: {off_diag_vals.min():.3f}, Max: {off_diag_vals.max():.3f})")


🔗 Mean Inter-Asset Correlation: 0.560 (Min: 0.206, Max: 0.819)


## 3. Fit Diebold-Yilmaz VAR Model & Variance Decomposition (GFEVD)

We fit a Vector Autoregressive model $\text{VAR}(p=2)$ on the daily log return series:
$$r_t = c + \sum_{l=1}^p \Phi_l r_{t-l} + \epsilon_t, \quad \epsilon_t \sim \mathcal{N}(0, \Sigma)$$

We then compute the **Generalized Forecast Error Variance Decomposition (GFEVD)** over a forecast horizon $H = 10$ days:
$$\theta_{ij}(H) = \frac{\sigma_{jj}^{-1} \sum_{h=0}^{H-1} (e_i' A_h \Sigma e_j)^2}{\sum_{h=0}^{H-1} (e_i' A_h \Sigma A_h' e_i)}$$

Where $\theta_{ij}(H)$ measures the proportion of asset $i$'s future return uncertainty explained by shocks originating from asset $j$.


In [8]:
baseline = VARSpilloverBaseline(maxlags=2, forecast_horizon=10)
baseline.fit(ret_active)

spillover_results = baseline.compute_spillover_table()
tsi = spillover_results["total_spillover_index"]
directional_table = spillover_results["directional_table"]
net_spillover = spillover_results["net_spillover"].sort_values(ascending=False)

print("════════════════════════════════════════════════════════════════════")
print(f"📊 DIEBOLD-YILMAZ TOTAL SPILLOVER INDEX (TSI): {tsi:.2f}%")
print("════════════════════════════════════════════════════════════════════")
print(f"Interpretation: {tsi:.1f}% of total return forecast variance in the crypto market")
print("is attributable to cross-asset contagion, with the remainder due to idiosyncratic shocks.")


[2026-09-30 00:18:49] [INFO] [cryptograph.models.baselines.var_spillover] - Fitted VAR baseline with lag order: 2
════════════════════════════════════════════════════════════════════
📊 DIEBOLD-YILMAZ TOTAL SPILLOVER INDEX (TSI): 54.24%
════════════════════════════════════════════════════════════════════
Interpretation: 54.2% of total return forecast variance in the crypto market
is attributable to cross-asset contagion, with the remainder due to idiosyncratic shocks.


c:\Users\daksh\AppData\Local\Programs\Python\Python311\Lib\site-packages\statsmodels\tsa\base\tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


## 4. Directional Spillover Table (From Others vs. To Others)

The full Diebold-Yilmaz table summarizes the directional shock transmission budget across all market participants.


In [9]:
display(directional_table.round(2))


,BTC,ETH,BNB,SOL,XRP,DOGE,ADA,AVAX,LINK,DOT,...,BCH,LTC,NEAR,XLM,ETC,ATOM,XMR,FIL,SHIB,FROM
BTC,97.15,0.14,0.31,0.19,0.53,0.22,0.13,0.12,0.30,0.18,...,0.20,0.04,0.08,0.01,0.04,0.05,0.01,0.03,0.06,2.85
ETH,64.60,32.15,0.22,0.44,0.58,0.44,0.08,0.04,0.23,0.09,...,0.20,0.13,0.23,0.11,0.04,0.04,0.09,0.04,0.10,67.85
BNB,41.74,5.47,48.33,0.48,0.14,0.86,0.03,0.17,0.80,0.05,...,0.31,0.36,0.29,0.01,0.11,0.19,0.01,0.25,0.26,51.67
SOL,37.38,9.23,3.03,47.59,0.02,0.16,0.08,0.07,0.13,0.14,...,0.73,0.04,0.39,0.06,0.07,0.19,0.07,0.06,0.07,52.41
XRP,35.68,5.51,1.39,1.14,51.73,1.77,0.12,0.04,0.62,0.11,...,0.27,0.16,0.11,0.09,0.43,0.04,0.20,0.11,0.22,48.27
DOGE,28.25,2.35,0.09,0.42,0.59,65.67,0.23,0.01,0.12,0.40,...,0.47,0.03,0.09,0.09,0.42,0.07,0.53,0.04,0.03,34.33
ADA,45.03,7.76,1.27,2.51,4.54,0.79,35.49,0.02,0.53,0.03,...,0.23,0.02,0.61,0.05,0.07,0.03,0.43,0.10,0.09,64.51
AVAX,39.22,7.04,2.92,3.54,1.64,0.46,3.90,39.66,0.31,0.02,...,0.35,0.02,0.10,0.02,0.15,0.00,0.06,0.28,0.15,60.34
LINK,47.79,12.90,1.38,1.40,2.10,0.48,1.85,0.70,29.23,0.45,...,0.26,0.22,0.10,0.09,0.07,0.36,0.06,0.01,0.21,70.77
DOT,47.22,10.59,2.31,1.57,2.17,0.80,3.87,1.30,1.71,27.14,...,0.29,0.02,0.18,0.15,0.16,0.01,0.18,0.02,0.14,72.86


## 5. Net Systemic Connectedness Ranking (Transmitters vs. Receivers)

The **Net Spillover Index** is defined as:
$$\mathcal{S}_i = \text{Spillover To Others}_i - \text{Spillover From Others}_i$$
- $\mathcal{S}_i > 0$: **Net Transmitter** of systemic risk (e.g. Bitcoin, Ethereum).
- $\mathcal{S}_i < 0$: **Net Receiver** / consumer of external shocks (e.g. Solana, Avalanche).


In [10]:
net_df = pd.DataFrame({
    "Asset": net_spillover.index,
    "Net_Spillover": net_spillover.values,
    "Role": ["Net Transmitter" if v > 0 else "Net Receiver" for v in net_spillover.values]
})

fig_net = px.bar(
    net_df,
    x="Asset",
    y="Net_Spillover",
    color="Role",
    color_discrete_map={"Net Transmitter": "#00e676", "Net Receiver": "#ff1744"},
    title="Net Systemic Connectedness Ranking (Diebold-Yilmaz 2012 Baseline)",
    labels={"Net_Spillover": "Net Spillover Index (%)", "Asset": "Cryptocurrency"}
)
fig_net.update_layout(height=450, xaxis_tickangle=-45)
fig_net.show()

print("Top 3 Net Transmitters (Systemic Drivers):")
for a, v in net_spillover.head(3).items():
    print(f"  ▲ {a:5s}: +{v:6.2f}%")

print("\nTop 3 Net Receivers (Vulnerable Victims):")
for a, v in net_spillover.tail(3).items():
    print(f"  ▼ {a:5s}: {v:6.2f}%")


Top 3 Net Transmitters (Systemic Drivers):
  ▲ BTC  : +731.50%
  ▲ ETH  : + 52.80%
  ▲ XRP  : + -3.92%

Top 3 Net Receivers (Vulnerable Victims):
  ▼ ETC  : -66.86%
  ▼ LTC  : -67.24%
  ▼ XLM  : -68.60%


## 6. Export Econometric Connectedness Benchmark for Deep Learning Validation

We export the fitted net connectedness series to [`data/processed/statistical_connectedness.csv`](../data/processed/statistical_connectedness.csv). In **Notebook 03**, `ExplanationValidator` uses this file to compute the **Spearman Rank Correlation** against the node attention scores learned by `TempME` to evaluate alignment between econometric and deep learning explanations.


In [11]:
out_path = Path("../data/processed/statistical_connectedness.csv")
spillover_df = pd.DataFrame({
    "Asset": net_spillover.index,
    "Net_Spillover": net_spillover.values
})
spillover_df.to_csv(out_path, index=False)

print("════════════════════════════════════════════════════════════════════")
print(f"💾 Ground-truth statistical connectedness exported to:")
print(f"   {out_path.resolve()}")
print(f"✅ Verified {len(spillover_df)} asset rankings ready for Notebook 03!")
print("════════════════════════════════════════════════════════════════════")


════════════════════════════════════════════════════════════════════
💾 Ground-truth statistical connectedness exported to:
   D:\GitHubProjects\CryptoGraph\data\processed\statistical_connectedness.csv
✅ Verified 20 asset rankings ready for Notebook 03!
════════════════════════════════════════════════════════════════════


## 6. Export Econometric Connectedness Benchmark for Deep Learning Validation

We export the fitted net connectedness series to [`data/processed/statistical_connectedness.csv`](../data/processed/statistical_connectedness.csv). In **Notebook 03**, `ExplanationValidator` uses this file to compute the **Spearman Rank Correlation** against the node attention scores learned by `TempME` to evaluate alignment between econometric and deep learning explanations.
